# <font color="#800080">☔ REPRESA Moçambique: Análise e Avaliação de Precipitação de Ciclones Tropicais</font>

Materiais de formação para análise e avaliação de conjuntos de dados de precipitação de Ciclones Tropicais em Moçambique.

**Desenvolvido por:** Dra. Helen Hooker, Universidade de Reading  **Projecto:** REPRESA

---

### 🚀 **BEM-VINDO! Por favor leia estas instruções importantes primeiro** 🚀

Este caderno foi concebido para uma experiência de formação interactiva usando o **Google Colab**. Não precisa de instalar nada no seu computador.

#### Antes de Começar:
✅ **Precisa de uma conta Google** (conta Gmail gratuita)  
✅ Entre no [Google Drive](https://drive.google.com) antes do workshop

---

### **PASSO 1️⃣: Abra a sua própria cópia editável** 🔑

**⚠️ IMPORTANTE:** Este caderno normalmente abre em **modo só de leitura**. Para guardar o seu trabalho e executar código, **DEVE** guardar a sua própria cópia:

1.  Clique em **"Ficheiro"** na barra de menu do Colab.
2.  Seleccione **"Guardar uma cópia no Drive"**.
3.  A sua cópia pessoal editável abrirá automaticamente.
4.  Todas as suas alterações serão então guardadas automaticamente no seu Google Drive (numa pasta chamada "Colab Notebooks").

---

### **PASSO 2️⃣: Execute as células uma por uma** ▶️

Depois de ter a sua própria cópia, por favor execute as células de código uma por uma de cima para baixo.

*   **Clique no botão "Play" (▶️)** nos parênteses rectos `[ ]` no lado esquerdo de cada célula de código.
*   **OU Use o Atalho de Teclado:** Seleccione a célula e pressione `Shift + Enter` (ou `Shift + Return`).

---

### 3️⃣ **PASSO 3: Escolha uma tempestade e descarregue os dados** ⬇️

Assim que estiver na sua cópia editável e souber como executar células:

1.  Execute as primeiras 3 células de código no seu caderno.
2.  Será solicitado a seleccionar a tempestade que deseja analisar (ex: `FREDDY`).
3.  Os dados da tempestade escolhida serão descarregados automaticamente.

---

## 📚 Visão Geral do Conteúdo do Workshop

Este workshop concentra-se na avaliação do novo **conjunto de dados de precipitação CCAM** em escala quilométrica em comparação com outros produtos de precipitação comummente usados, incluindo:
- **ERA5** (Reanálise ECMWF)
- **GPM IMERG** (Precipitação por satélite)
- **Observações de pluviómetros** (Pluviómetros locais)

Precipitação total da tempestade, precipitação diária e horária estão disponíveis para quatro ciclones tropicais / tempestades impactantes:

* **CHEDZA 2015**
* **ANA 2022**
* **GOMBE 2022**
* **FREDDY 2023**

<center>
  <img src="https://github.com/helenhooker/REPRESA_Mozambique_TC_rainfall/blob/c71e943f8fd28cf096ef30596808a117c7aaf5c9/4_storms_tracks.png?raw=1" alt="Trajectórias das Tempestades" width="500"/>
</center>

Os novos dados CCAM estão disponíveis em alta resolução (tamanho de grade de 4 km) e em resolução ERA5 (tamanho de grade de 25 km, rotulado LR). A precipitação por satélite também está disponível em resolução nativa (10 km) e em resolução ERA5.

## 📚 Resultados de Aprendizagem

* Comparar a precipitação CCAM com conjuntos de dados de precipitação existentes, tanto espacialmente como ao longo do tempo.
* Investigar tempestades, regiões e períodos de tempo de interesse.
* Desenvolver competências de visualização e avaliação de dados em Python e Jupyter Notebooks.

<span style="color: blue;">Dra. Helen Hooker, Universidade de Reading</span>
![LOGO](https://github.com/helenhooker/REPRESA_Mozambique_TC_rainfall/blob/main/REPRESA_logo.png?raw=1)




<font color="#800080" size="5">1. Carregar dados</font>  
<font color="#800080" size="4">1.1 Importar módulos</font>  

Adicione mais módulos aqui conforme necessário...  


In [ ]:
!pip install cartopy
import cartopy

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import pandas as pd
import os
import sys

<font color="#800080" size="4">1.2 Escolher uma tempestade e descarregar dados</font>

Os conjuntos de dados de precipitação para quatro tempestades estão guardados num Google Drive. Execute a célula para selecionar uma tempestade e descarregar os dados.

In [ ]:
# --- 1. Define the Storm Options ---

available_storms = ["FREDDY", "ANA", "CHEDZA", "GOMBE"]

# --- 2. Check Environment & Setup Data ---
if 'google.colab' in str(get_ipython()):
    print("🚀 Detected Google Colab environment. Setting up data.")
    import gdown # Library to download from Drive

    # --- FILE IDs ---
    google_drive_file_ids = {
        "ANA": "1ZDu8pEgw6L53VT7-k1b-QxBhaP3S0qk3",
        "FREDDY": "1ONV4X3nDYFp7R5WUp6Nq4CrCIcl71Qe3",
        "CHEDZA": "1i35Y5Cm3ARukFovH5lwjQu88r1JSAEL_",
        "GOMBE": "1HctO0zTB8yTnS9mW5cGRajeXqV6JOBrt"
    }
    # ----------------------------------

    # A. Always ask the user which storm they want to analyse when this cell is executed
    print(f"Available storms: {', '.join(available_storms)}")
    # The 'input()' function will always prompt, allowing the user to change the storm.
    storm_name = input("Enter the storm name you want to analyse (e.g., FREDDY): ").strip().upper()

    if storm_name not in available_storms:
        print(f"❌ Error: '{storm_name}' is not in the list. Please re-run this cell and choose from available options.")
        sys.exit() # Stop execution if wrong name

    # B. Get the Secret File ID from the dictionary
    file_id = google_drive_file_ids.get(storm_name)

    if not file_id:
        print(f"❌ Error: File ID not found for {storm_name}. Please check the 'google_drive_file_ids' dictionary.")
        sys.exit()

    # C. Download and Unzip
    zip_name = f"{storm_name}.zip"

    # Check if the folder for the *currently selected storm* exists
    if not os.path.exists(storm_name):
        print(f"\n⬇️ Downloading {storm_name} data...")
        try:
            # Download from Google Drive using the ID
            gdown.download(id=file_id, output=zip_name, quiet=False)

            print("📦 Unzipping...")
            get_ipython().system(f'unzip -q {zip_name}')
            print("✅ Done! Data is ready.")

            # Clean up zip file
            get_ipython().system(f'rm {zip_name}')
        except Exception as e:
            print(f"❌ Download failed. Please check the File ID. Error: {e}")
            sys.exit()
    else:
        print(f"✅ Data for {storm_name} is already unzipped.")

    # D. Set the Working Directory for Colab
    working_dir = "/content"

else:
    # --- Local / JASMIN Environment Setup ---
    print("💻 Detected Local/JASMIN environment.")

    # For local/JASMIN, you might hardcode FREDDY, or allow selection via input()
    storm_name = 'FREDDY' # <<< --- Default for local testing ---
    working_dir = "/home/users/hmhooker/Madagascar/cyclone_rainfall_data/" # <<< --- Adjust this for your local setup ---
    print(f"Using local path: {working_dir}")

print(f"\n🌍 Setup Complete. Analyzing storm: {storm_name}")
print(f"📂 Working Directory: {working_dir}")


<font color="#800080" size="4">1.3 Extrair precipitação total da tempestade para a tempestade escolhida</font>

In [ ]:
# The variables 'storm_name' and 'working_dir' are already set by the cell above!

print(f"Importing data for {storm_name}...")

# Import data

CCAM = xr.open_dataset(f'{working_dir}/{storm_name}/CCAM_rainfall/CCAM_TOTAL_{storm_name}.nc')

CCAM_LR = xr.open_dataset(f'{working_dir}/{storm_name}/CCAM_rainfall/CCAM_LR_TOTAL_{storm_name}.nc')

ERA5 = xr.open_dataset(f'{working_dir}/{storm_name}/ERA5_rainfall/ERA5_TOTAL_{storm_name}.nc')

IMERG = xr.open_dataset(f'{working_dir}/{storm_name}/IMERG_rainfall/GPM_TOTAL_{storm_name}.nc')

# Complete the path

#IMERG_LR = xr...

gauge_data = pd.read_csv(f'{working_dir}/{storm_name}/{storm_name}_gauge.csv')

<font color="#800080" size="5">2. Inspecionar os dados</font>

<font color="#800080" size="4">2.1 Verificar os conjuntos de dados</font>

In [ ]:
# Change to inspect dataset of choice

#CCAM
#CCAM_LR
ERA5
#IMERG
#IMERG_LR
#gauge_data

<font color="#800080" size="4">2.2 Extrair variáveis e recortar dados</font>

Pode notar que os dados IMERG estão transpostos em comparação com outros conjuntos de dados, por isso corrigimos isso aqui (lembre-se de corrigir também o IMERG_LR!)

In [ ]:
IMERG['precipitation'] = IMERG['precipitation'].transpose('lat', 'lon')

# Complete for IMERG_LR

#IMERG_LR...

<font color="#800080" size="4">2.3 Escolher uma região do seu interesse</font>

In [ ]:
lat_range = slice(-35, -5.0)
lon_range = slice(28, 58)

# Crop the rainfall data and pre-process so that all are in mm
CCAM_rainfall = CCAM['rnd'].sel(lat = lat_range, lon = lon_range) / 24 # convert mm/day to mm/h

#CCAM_LR_rainfall =
CCAM_LR_rainfall = CCAM_LR['rnd_LR'].sel(latitude = lat_range, longitude = lon_range) / 24

ERA5_rainfall = ERA5['tp'].where((ERA5['latitude'] >= lat_range.start) & (ERA5['latitude'] <= lat_range.stop) &
                                     (ERA5['longitude'] >= lon_range.start) & (ERA5['longitude'] <= lon_range.stop),
                                     drop=True) * 1000 # convert m to mm

IMERG_rainfall = IMERG['precipitation'].sel(lat = lat_range, lon = lon_range)

# Complete...
#IMERG_LR_rainfall =

<font color="#800080" size="4">2.4 Verificar se as precipitações máximas e mínimas são realistas</font>

1.   Item da lista
2.   Item da lista

In [ ]:
CCAM_rainfall.max() # can you check the min too...

<font color="#800080" size="5">3. Plotar os dados</font>

<font color="#800080" size="4">3.1 Gráficos básicos rápidos</font>

Um gráfico rápido pode ser uma boa verificação inicial dos dados. Altere o código para visualizar os outros conjuntos de dados.

Mude o nome da tempestade e dê uma vista de olhos rápida a outra tempestade também.

In [ ]:
CCAM_rainfall.plot()

<font color="#800080" size="4">3.2 Plotar os dados de pluviómetros</font>

Tente modificar a precipitação de pluviómetro plotada para mostrar precipitação diária num dia interessante e para uma tempestade diferente.

In [ ]:
# Filter the data for the specific date of interest format e.g. 13/03/2023

date_of_interest = 'TOTAL'

# Define longitude and latitude ranges
lat_range = slice(-5, -35)
lon_range = slice(28, 58)

# Create a plot with Cartopy
fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_extent([lon_range.start, lon_range.stop, lat_range.start, lat_range.stop])

# Add country borders
ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=1, edgecolor='black')

# Add coastlines
ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor='black')

# Set the face color of the ocean to light blue
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')

# Add latitude and longitude gridlines
ax.gridlines(draw_labels=True, linestyle='--', linewidth=0.5, color='black', alpha=0.7)

gauge_rainfall = gauge_data[['STATION', 'LONGIT', 'LATIT', date_of_interest]]

gauge_rainfall.loc[:, date_of_interest] = pd.to_numeric(gauge_rainfall[date_of_interest], errors='coerce')

# Drop rows with missing latitude or longitude values
gauge_rainfall = gauge_rainfall.dropna(subset=['LONGIT', 'LATIT'])

# Define discrete intervals for the color scale
boundaries = np.arange(0, gauge_rainfall[date_of_interest].max() + 50, 50)  # 0 to max + 50 in steps of 50
norm = BoundaryNorm(boundaries, ncolors=256, clip=True)

# Plot each point on the map with normalized colors
scatter = ax.scatter(
    gauge_rainfall['LONGIT'],
    gauge_rainfall['LATIT'],
    c=gauge_rainfall[date_of_interest],
    cmap='viridis',
    norm=norm,
    edgecolors='black',
    s=30,
    transform=ccrs.PlateCarree()
)

# Add colorbar with discrete intervals
cbar = plt.colorbar(scatter, ax=ax, orientation='vertical', fraction=0.03, pad=0.1)
cbar.set_label('Rainfall (mm)')
cbar.set_ticks(boundaries)  # Set the ticks to match the discrete intervals
cbar.update_ticks()  # Refresh the ticks

# Set title
title = f'{storm_name}:{date_of_interest} Rainfall'
ax.set_title(title)

# Show the plot
plt.show()

<font color="#800080" size="4">3.3 Plotar dados espaciais</font>

Aqui vai plotar três conjuntos de dados em conjunto para comparar.

Mude CCAM e IMERG para CCAM_LR e IMERG_LR, o que nota?

Consegue melhorar e criar uma barra de cores personalizada?

In [ ]:
# Find the overall maximum value across the datasets
max_value = max(CCAM_rainfall.max().values, ERA5_rainfall.max().values, IMERG_rainfall.max().values)

# Define rounded discrete levels for the color bar
levels = np.arange(100, max_value + 1, 100)

# Create a figure with subplots for side-by-side comparison
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 8), subplot_kw={'projection': ccrs.PlateCarree()})

# Add features and grid lines
for index, ax in enumerate(axes):
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=1, edgecolor='black')
    ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor='black')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue', zorder=2)
    ax.add_feature(cfeature.LAND, facecolor='white')
    ax.set_extent([30, 52, -30, -10])  # Adjust based on your area of interest

    # Configure gridlines
    gl = ax.gridlines(linewidth=0.5, color='gray', alpha=0.5, linestyle='--', zorder=3)
    gl.right_labels = False  # Disable right labels for all plots
    if index == 0:
        gl.left_labels = True   # Enable left labels only for the first axis (left plot)
        gl.bottom_labels = True

# Plot CCAM
ccam_plot = CCAM_rainfall.fillna(0).where(CCAM_rainfall > 100).plot(ax=axes[0], cmap='viridis', levels=levels, add_colorbar=False)
axes[0].set_title(f'{storm_name}: CCAM rainfall')
#axes[0].set_xlabel('Longitude')
#axes[0].set_ylabel('Latitude')

# Plot ERA5
era5_plot = ERA5_rainfall.fillna(0).where(ERA5_rainfall > 100).plot(ax=axes[1], cmap='viridis', levels=levels, add_colorbar=False)
axes[1].set_title(f'{storm_name}: ERA5 rainfall')


# Plot IMERG
imerg_plot = IMERG_rainfall.fillna(0).where(IMERG_rainfall > 100).plot(ax=axes[2], cmap='viridis', levels=levels, add_colorbar=False)
axes[2].set_title(f'{storm_name}: IMERG rainfall')

# Add a single colorbar for all three plots
cbar = plt.colorbar(ccam_plot, ax=axes, orientation='vertical', pad=0.02, aspect=20, shrink=0.55)
cbar.set_label('Rainfall (mm)')
cbar.set_ticks(levels)

plt.show()

<font color="#800080" size="4">3.4 Comparar precipitação de pluviómetros com conjuntos de dados espaciais</font>

Aqui vai extrair a precipitação da célula de grade correspondente a cada pluviómetro e plotar estes num gráfico de dispersão junto com indicadores de desempenho: erro absoluto médio (MAE), erro quadrático médio (RMSE) e coeficiente de correlação de Pearson (PCC). Experimente isto para diferentes conjuntos de dados e tempestades.

Quais são as diferenças ou semelhanças entre os conjuntos de dados e tempestades, consegue explicar porquê?

In [ ]:
# Change the rainfall dataset
rainfall_dataset = CCAM_rainfall

dataset_rainfall = []
for i, row in gauge_rainfall.iterrows():
    lat, lon = row['LATIT'], row['LONGIT']
    dataset_rainfall.append(rainfall_dataset.sel(lat=lat, lon=lon, method='nearest').item())

# Create a scatter plot to compare observed and modeled precipitation
fig, ax = plt.subplots()
ax.scatter(gauge_rainfall[date_of_interest], dataset_rainfall, color='purple')
ax.plot([0, gauge_rainfall[date_of_interest].max()], [0, gauge_rainfall[date_of_interest].max()], color='black', linestyle='--', label='1:1 Line')
ax.set_xlabel('Gauge rainfall (mm)')
ax.set_ylabel('Rainfall (mm)')
ax.set_title(f'{storm_name} total rainfall')

gauge_values = np.array(gauge_rainfall[date_of_interest])
dataset_values = np.array(dataset_rainfall)

# Calculate MAE and RMSE
mae_value = np.mean(np.abs(gauge_values - dataset_values))
rmse_value = np.sqrt(np.mean((gauge_values - dataset_values) ** 2))
corr_coef = np.corrcoef(gauge_values, dataset_values)[0, 1]

# Define units
units = 'mm'

# Add MAE and RMSE with units to the plot
ax.annotate(f'MAE: {mae_value:.2f} {units}', xy=(0.75, 0.95), xycoords='axes fraction', ha='right', fontsize=10, color='black')
ax.annotate(f'RMSE: {rmse_value:.2f} {units}', xy=(0.75, 0.9), xycoords='axes fraction', ha='right', fontsize=10, color='black')
ax.annotate(f'PCC: {corr_coef:.2f}', xy=(0.75, 0.85), xycoords='axes fraction', ha='right', fontsize=10, color='black')

plt.show()

<font color="#800080" size="4">3.5 Calcular a diferença espacial</font>

Aqui pode escolher quais conjuntos de dados comparar calculando a diferença na precipitação. Para calcular a diferença espacial, os conjuntos de dados devem estar na mesma resolução horizontal (tamanho de grade), pelo que precisará de usar CCAM_LR_rainfall e IMERG_LR_rainfall para comparar os três conjuntos de dados.

Onde estão as principais diferenças? Por que razão existem diferenças em locais particulares?

In [ ]:
# Calculate the rainfall difference between two datasets
rainfall_difference = CCAM_LR_rainfall - ERA5_rainfall
rainfall_difference.max()

In [ ]:
# Create a plot
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_extent([lon_range.start, lon_range.stop, lat_range.start, lat_range.stop])

# Add country borders
ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=1, edgecolor='black')

# Add coastlines
ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor='black')

ax.add_feature(cfeature.OCEAN, facecolor='lightblue', zorder = 10)

# Plot rainfall difference data
pcm = rainfall_difference.fillna(0).plot(ax=ax, transform=ccrs.PlateCarree(), cmap='seismic_r', add_colorbar=False, vmin=-1000, vmax=1000)

# Add latitude and longitude gridlines
ax.gridlines(draw_labels=True, linestyle='--', linewidth=0.5, color='black', alpha=0.7)

# Add colorbar
cbar = plt.colorbar(pcm, ax=ax, orientation='vertical', fraction=0.03, pad=0.15)
cbar.set_label('Rainfall difference (mm)')

# Customize plot
ax.set_title(f'{storm_name}: CCAM_LR - ERA5')

# Show the plot
plt.show()

<font color="#800080" size="5">4. A sua própria análise ☔</font>

O que mais gostaria de descobrir? Continue a investigar os dados aqui com o que aprendeu até agora e desenvolvendo o seu próprio código. Sinta-se à vontade para usar ferramentas de IA como o ChatGPT para ajudar!

Consegue plotar dados de precipitação diária ou horária num mapa para dias ou horas selecionados para diferentes tempestades e conjuntos de dados?

Consegue plotar uma série temporal diária ou horária de precipitação (hietogramas) para um local escolhido (célula de grade, área ou sub-bacia) e período de tempo para um ou mais conjuntos de dados para comparar?

Consegue comparar a distribuição de precipitação total, diária e horária em cada conjunto de dados? Como se comparam os extremos de precipitação?